In [1]:
"""
Download S&P 500 daily OHLC data.

Construct volatility targets:

Returns:
    - daily log returns
    - absolute returns

Realized volatility:
    - 5-day annualized rolling volatility
    - 10-day annualized rolling volatility
    - 21-day annualized rolling volatility

OHLC volatility:
    - Garman-Klass volatility
    - Yang-Zhang volatility

Outputs:
    sp500_daily.csv
"""

import os
import numpy as np
import pandas as pd
import yfinance as yf


START_DATE = "2022-01-01"
END_DATE   = "2026-05-31"

SP500_TICKER = "^GSPC"

DATA_DIR = (
    "C:/Python/CSUREMM/test"
)

os.makedirs(
    DATA_DIR,
    exist_ok=True
)


# ------------------------------------------------
# Download
# ------------------------------------------------

def download(
    ticker,
    start,
    end
):

    print(
        f"[yfinance] downloading {ticker}"
    )

    df = yf.download(
        ticker,
        start=start,
        end=end,
        auto_adjust=False,
        progress=False
    )


    df.index = (
        pd.to_datetime(df.index)
        .normalize()
    )

    df.index.name = "date"


    # yfinance multi-index fix
    if isinstance(df.columns, pd.MultiIndex):

        df.columns = (
            df.columns
            .get_level_values(0)
        )


    return df



# ------------------------------------------------
# Volatility estimators
# ------------------------------------------------


def garman_klass(
    df,
    window
):

    """
    Garman-Klass volatility estimator.

    Uses:
    Open High Low Close

    """

    log_hl = np.log(
        df["High"] /
        df["Low"]
    )


    log_co = np.log(
        df["Close"] /
        df["Open"]
    )


    rs = (
        0.5 * log_hl**2
        -
        (2*np.log(2)-1)
        *
        log_co**2
    )


    return np.sqrt(
        rs
        .rolling(window)
        .mean()
        *
        252
    )



def yang_zhang(
    df,
    window
):

    """
    Yang-Zhang volatility estimator.

    Combines:
    - overnight volatility
    - open-close volatility
    - Rogers-Satchell volatility
    """

    log_oc = np.log(
        df["Open"] /
        df["Close"].shift(1)
    )


    log_co = np.log(
        df["Close"] /
        df["Open"]
    )


    log_ho = np.log(
        df["High"] /
        df["Open"]
    )


    log_lo = np.log(
        df["Low"] /
        df["Open"]
    )


    rs = (
        log_ho *
        (log_ho - log_co)
        +
        log_lo *
        (log_lo - log_co)
    )


    overnight = (
        log_oc
        .rolling(window)
        .var()
    )


    open_close = (
        log_co
        .rolling(window)
        .var()
    )


    rs_var = (
        rs
        .rolling(window)
        .mean()
    )


    k = 0.34 / (
        1.34 +
        (window+1)/(window-1)
    )


    yz = (
        overnight
        +
        k * open_close
        +
        (1-k)*rs_var
    )


    return np.sqrt(
        yz * 252
    )



# ------------------------------------------------
# Feature construction
# ------------------------------------------------


def build_sp500_features(df):


    out = pd.DataFrame(
        index=df.index
    )


    close = df["Close"]


    # prices

    out["sp500_close"] = close
    out["sp500_open"] = df["Open"]
    out["sp500_high"] = df["High"]
    out["sp500_low"] = df["Low"]
    out["sp500_volume"] = df["Volume"]



    # --------------------------------
    # returns
    # --------------------------------

    log_ret = np.log(
        close /
        close.shift(1)
    )


    out["sp500_log_return_1d"] = log_ret


    out["sp500_abs_return"] = (
        log_ret.abs()
    )



    out["sp500_return_5d"] = (
        close
        .pct_change(5)
    )



    out["sp500_return_21d"] = (
        close
        .pct_change(21)
    )



    # --------------------------------
    # forward targets
    # --------------------------------

    out["sp500_fwd_return_1d"] = (
        log_ret.shift(-1)
    )


    out["sp500_fwd_return_5d"] = (
        np.log(
            close.shift(-5) /
            close
        )
    )



    # --------------------------------
    # realized volatility
    # --------------------------------

    for w in [5,10,21]:

        out[
            f"sp500_rv{w}"
        ] = (
            log_ret
            .rolling(w)
            .std()
            *
            np.sqrt(252)
        )



    # --------------------------------
    # OHLC volatility
    # --------------------------------

    out["sp500_gk5"] = garman_klass(
        df,
        5
    )


    out["sp500_gk10"] = garman_klass(
        df,
        10
    )


    out["sp500_yz5"] = yang_zhang(
        df,
        5
    )


    out["sp500_yz10"] = yang_zhang(
        df,
        10
    )



    return out



# ------------------------------------------------
# Main
# ------------------------------------------------


def main():

    raw = download(
        SP500_TICKER,
        START_DATE,
        END_DATE
    )


    sp500 = build_sp500_features(
        raw
    )


    output = (
        DATA_DIR +
        "/sp500_daily.csv"
    )


    sp500.to_csv(
        output
    )


    print(
        f"[save] {output}"
    )


    print(
        sp500[
            [
                "sp500_close",
                "sp500_log_return_1d",
                "sp500_abs_return",
                "sp500_rv5",
                "sp500_yz10"
            ]
        ]
        .tail()
    )



if __name__ == "__main__":
    main()

[yfinance] downloading ^GSPC
[save] C:/Python/CSUREMM/test/sp500_daily.csv
            sp500_close  sp500_log_return_1d  sp500_abs_return  sp500_rv5  \
date                                                                        
2026-05-22  7473.470215             0.003720          0.003720   0.101085   
2026-05-26  7519.120117             0.006090          0.006090   0.102134   
2026-05-27  7520.359863             0.000165          0.000165   0.065719   
2026-05-28  7563.629883             0.005737          0.005737   0.040528   
2026-05-29  7580.060059             0.002170          0.002170   0.039386   

            sp500_yz10  
date                    
2026-05-22    0.105168  
2026-05-26    0.109332  
2026-05-27    0.103127  
2026-05-28    0.098604  
2026-05-29    0.097818  
